## CRC Proteomics Analysis

## Overview

This project analyzes colorectal cancer (CRC) proteomics data to identify proteins that distinguish tumor tissue from normal tissue. Using mass spectrometry-based protein expression data, I explore patterns in the dataset with PCA and build a logistic regression model to classify tumor vs normal samples. The project highlights proteins most influential in tumor classification.

## Goals

Visualize tumor and normal samples using PCA.

Train a logistic regression model to classify tumor vs normal tissue.

Identify the top 20 proteins most associated with tumor status.


## Tools

Python 3

pandas, numpy

scikit-learn (StandardScaler, PCA, LogisticRegression, train_test_split, metrics)

matplotlib, seaborn for visualization

Jupyter Notebook / Google Colab

## Install Packages & Imports

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve


## Load Datasets

## Inspect Data

In [ ]:
import pandas as pd

expression_df = pd.read_csv('/content/proteinGroups_ppb_final.txt', sep='\t', index_col=0)
sample_info_df = pd.read_csv('/content/Sample_classification.txt', sep='\t')


In [ ]:
print("Expression matrix shape:", expression_df.shape)
print("Sample info columns:", sample_info_df.columns)
print(sample_info_df.head())


## Prepare Expression Matrix

In [ ]:
expression_df_samples = expression_df.T

expression_df_samples.index = expression_df_samples.index.str.strip()
sample_info_df['Name'] = sample_info_df['Name'].str.strip()

common_samples = expression_df_samples.index.intersection(sample_info_df['Name'])
expression_df_samples = expression_df_samples.loc[common_samples]
sample_info_df = sample_info_df[sample_info_df['Name'].isin(common_samples)]

labels = sample_info_df.set_index('Name')['Tissue'].map(lambda x: 1 if x=='Tumor' else 0)
print(labels.value_counts())




## Standardize Data

In [ ]:
expression_df_samples_clean = expression_df_samples.dropna(axis=1, how='all')


imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(expression_df_samples_clean)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)




## PCA Visualization

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=labels, palette={0:'blue',1:'red'}, legend='full')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA of Samples')
plt.show()



## Split Data & Train Logistic Regression

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, labels, test_size=0.2, random_state=42, stratify=labels
)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)



## Evaluate Model and Plot ROC Curve

In [ ]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
print("Test Accuracy:", accuracy)
print("ROC AUC:", roc_auc)

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0,1],[0,1],'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()


## Identify Top 20 Proteins

In [ ]:
coef = clf.coef_[0]
top_idx = np.argsort(np.abs(coef))[::-1][:20]
top_features = expression_df_samples_clean.columns[top_idx]
top_coefs = coef[top_idx]

top_features_df = pd.DataFrame({
    'Protein': top_features,
    'Coefficient': top_coefs
})

print(top_features_df)

plt.figure(figsize=(8,6))
sns.barplot(x='Coefficient', y='Protein', data=top_features_df)
plt.title('Top 20 Proteins by Logistic Regression Coefficient')
plt.show()



Conclusion

The CRC proteomics dataset was successfully cleaned and aligned, retaining 143 Tumor and 131 Non-tumor samples. Tissue types were mapped to binary labels, and only samples present in both the expression matrix and sample information were kept. Dimensionality reduction (PCA) and logistic regression modeling were performed, identifying the top 20 proteins most associated with tumor status. The dataset is now ready for further downstream analysis.